# Assignment 1: Classify Breast Cancer Cases

Author: Tobias Beekmans  
Master ICT – Software Engineering  
DataOps Specialisation Project – Individual Assignment  
Submission Date: 15.03.2026

In [37]:
# Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from breast_cancer_assignment.dataset import load_data

# Notebook settings
pd.set_option("display.max_columns", None)

# Load dataset
df = load_data()

# 3.0 Data Preparation

## 3.1 Feature Selection

Before training machine learning models, the dataset must be prepared by defining the predictor variables and the target variable.

The dataset contains numerical measurements describing morphological characteristics of cell nuclei. These measurements serve as the input features for the classification task.

The diagnostic label indicates whether a tumour is malignant or benign and is therefore used as the target variable.

In [38]:
# Define feature matrix and target variable

X = df.drop(columns="target")
y = df["target"]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

Feature matrix shape: (569, 30)
Target vector shape: (569,)


The resulting feature matrix contains 30 numerical predictor variables, while the target vector contains the binary diagnosis label.

At this stage, all original predictor variables are retained. Feature selection in the sense of removing redundant or less informative variables is not yet performed, since the initial goal is to preserve the full dataset structure for model comparison.

Potential feature redundancy identified in the data understanding phase will be taken into account later during modelling and evaluation.

## 3.2 Train–Test Split

To evaluate the performance of machine learning models objectively, the dataset is divided into separate training and test sets.

The training set is used to train the models, while the test set is reserved for evaluating model performance on previously unseen data. This separation helps prevent overly optimistic performance estimates.

A stratified split is applied to ensure that the class distribution of malignant and benign tumours remains consistent in both subsets.

In [39]:
# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (455, 30)
Test set: (114, 30)


The dataset is split into training and test sets using an 80/20 ratio.

Stratified sampling ensures that the proportion of malignant and benign cases remains consistent across both subsets. This is particularly important in classification problems where class imbalance may affect model performance.

The resulting training set will be used for model training and hyperparameter tuning, while the test set will serve as an independent dataset for final model evaluation.

## 3.3 Feature Scaling

The dataset contains numerical features with substantially different scales. For example, variables describing tumour size such as `area` and `perimeter` have much larger numerical values than variables such as `smoothness` or `fractal_dimension`.

Machine learning algorithms that rely on distance calculations or gradient optimisation can be sensitive to such differences in feature scale. Without scaling, variables with larger magnitudes may dominate the learning process.

To address this issue, feature scaling is applied using the **StandardScaler**, which standardises each feature to have a mean of 0 and a standard deviation of 1.

The scaler is fitted only on the training data to avoid data leakage. The same transformation is then applied to both the training and test sets.

In [40]:
# Initialize scaler
scaler = StandardScaler()

# Fit scaler on training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply same transformation to test data
X_test_scaled = scaler.transform(X_test)

print("Scaled training set shape:", X_train_scaled.shape)
print("Scaled test set shape:", X_test_scaled.shape)

# Convert scaled training data to DataFrame for inspection
scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
print(scaled_df.describe().round(2))

Scaled training set shape: (455, 30)
Scaled test set shape: (114, 30)
       mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
count       455.00        455.00          455.00     455.00           455.00   
mean         -0.00          0.00           -0.00      -0.00             0.00   
std           1.00          1.00            1.00       1.00             1.00   
min          -2.03         -2.17           -1.98      -1.47            -2.50   
25%          -0.70         -0.74           -0.70      -0.68            -0.72   
50%          -0.23         -0.10           -0.23      -0.31            -0.04   
75%           0.48          0.56            0.50       0.35             0.65   
max           4.02          4.55            4.02       5.37             3.61   

       mean compactness  mean concavity  mean concave points  mean symmetry  \
count            455.00          455.00               455.00         455.00   
mean              -0.00           -0.00            

The scaling process standardises the feature distributions by transforming them to a mean of approximately 0 and a standard deviation of approximately 1.

The inspection confirms that the scaling procedure was applied correctly. Performing the fit operation exclusively on the training data prevents information from the test set from influencing the transformation, thereby avoiding data leakage.

The scaled datasets will be used for training machine learning models in the subsequent modelling phase.

## 3.4 Final Dataset for Modelling

After completing the preparation steps, the dataset is ready for the modelling phase.

The original dataset has been divided into training and test sets to enable an unbiased evaluation of model performance. Feature scaling has been applied to ensure that all numerical variables operate on a comparable scale.

The resulting datasets are:

- **X_train_scaled**: scaled training feature matrix  
- **X_test_scaled**: scaled test feature matrix  
- **y_train**: training target labels  
- **y_test**: test target labels  

The training data will be used to train and tune machine learning models, while the test set will remain untouched during model development and will only be used for final performance evaluation.

These prepared datasets form the input for the modelling phase of the CRISP-DM process.

In [41]:
print("Training features:", X_train_scaled.shape)
print("Test features:", X_test_scaled.shape)
print("Training labels:", y_train.shape)
print("Test labels:", y_test.shape)

Training features: (455, 30)
Test features: (114, 30)
Training labels: (455,)
Test labels: (114,)


## 3.5 Export Processed Data

In [42]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

X_train_scaled_df.to_csv(output_dir / "X_train_scaled.csv", index=False)
X_test_scaled_df.to_csv(output_dir / "X_test_scaled.csv", index=False)

y_train.to_csv(output_dir / "y_train.csv", index=False)
y_test.to_csv(output_dir / "y_test.csv", index=False)